In [1]:
from ase import Atoms
from ase.io import read
from ase.build import molecule
from ase.units import Ha
from ase_dirac import DIRAC

Note that the current interface is rather minimal. We plan to add more functionality in next releases but of course also welcome user contributions (simply make a pull request if you have added something useful to this interface).

---

Added a small interface to be able to use DIRAC input files without the need of "translating" from DIRAC input to the dictionaries used by the ase_dirac.DIRAC calculator class:

In [2]:
xyz_string = """2
F2
O      0 0 0
O      0.00000       -0.75545     -0.47116
"""

fscc_str = """ **DIRAC
    .TITLE
    Molecular Te. DEA from the dianion.
    .WAVE F
    **HAMILTONIAN
    .LVCORR
    **WAVE FUNCTIONS
    .SCF
    .RELCCSD
    ! because we want this
    *SCF
    .PRINT 
    1
    !.CLOSED SHELL
    !54 52
    **RELCCSD
    .ENERGY
    
    
    .PRINT
    1
    .FOCKSP
    *CCFSPC
    .DOIE2
    .NACTH
    1 1 0 0
    .MAXIT
    200
    **MOLECULE
    *BASIS
    .DEFAULT
    dyall.3zp
    ! **Moltra
    ! .Active
    ! energy -10.0 20.0 1.0
    *END OF
"""

with open("molecule.xyz", "w") as f:
    f.write(xyz_string)

with open("dirac_example.inp", "w") as f:
    f.write(fscc_str)

In [3]:
ase_molecule = read("molecule.xyz", format="xyz")
ase_molecule.calc = DIRAC.from_file("dirac_example.inp")

# Also it can be called from string as:
# ase_molecule.calc = DIRAC.from_string(fscc_str)

param_list = [
    f"{key}, {ase_molecule.calc.parameters[key]}\n"
    for key in ase_molecule.calc.parameters.keys()
]

print("".join(param_list))

dirac, {'.TITLE': 'Molecular Te. DEA from the dianion.', '.WAVE F': '', '.4index': ''}
hamiltonian, {'.LVCORR': ''}
wave_function, {'.SCF': '', '.RELCCSD': '', '*SCF': {'.PRINT': '1'}, '**RELCCSD': {'.ENERGY': '', '.PRINT': '1', '.FOCKSP': ''}, '*CCFSPC': {'.DOIE2': '', '.NACTH': '1 1 0 0', '.MAXIT': '200'}}
molecule, {'*BASIS': {'.DEFAULT': 'dyall.3zp'}, '*END OF': {}, '*charge': {'.charge': '0'}, '*symmetry': {'.nosym': '#'}}
integrals, {'.nucmod': '1'}
moltra, {'.active': 'all'}



Where the default options in the `DIRAC.check_dirac_attributes()` method conflict with either inputted keywords or default `pam` parsing/default execution:
- `*charge: {'.charge': '0'}`: Occupation was established with `.CLOSED SHELL \n 54 52`. For the diatomic $O_2$ as proposed, the correct charge would be $-90$. This is not a big deal as occupation overrides charge. However this is redundant as the DIRAC parsing subroutine will default to $0$ in the absence of keyword.
- `*symmetry: {'.nosym': '#'}`: Linear symmetry was implicitly defined with `.CLOSED SHELL \n 54 52`. However, it is overriden, when `pam` would try to detect symmetry. This is a big deal as symmetry is ignored. 
- `integrals, {'.nucmod': '1'}` and `moltra, {'.active': 'all'}`: `moltra, {'.active': 'all'}` directly contradicts the default `MOLTRA` behaviour with `energy -10.0 20.0 1.0`. This can present issues in the correlation calculations, particularly in the `RELCCSD` module as it will be shown later, where the recalculation of the energy does not lead to the same values. 

In [4]:
# ase_molecule.get_potential_energy() / Ha

Also the multiplying the value by `Ha` when parsing to then later have to divide the energy by `Ha` to match the DIRAC output. 

```python
with open(output_filename, "r") as file:
        content = file.read()  # Read the entire content of the file

    ehfmatch = re.search(_scf_energy_re, content)
    if ehfmatch is not None:
        energy = float(ehfmatch.group(1)) * Ha
```

In [5]:
!grep "Vi  10   24    1g  " dirac.out -A  23

Which is not ideal. However, we can still use the template from the calculation to impose:

In [6]:
ase_molecule.calc.parameters.moltra.update({".active": "energy -10 20 1"})
ase_molecule.calc.parameters.molecule.update({"*symmetry": {".LINEAR": ""}})

param_list = [
    f"{key}, {ase_molecule.calc.parameters[key]}\n"
    for key in ase_molecule.calc.parameters.keys()
]

print("".join(param_list))

try:
    ase_molecule.calc.calculate()
except TypeError:
    ase_molecule.get_potential_energy() / Ha

dirac, {'.TITLE': 'Molecular Te. DEA from the dianion.', '.WAVE F': '', '.4index': ''}
hamiltonian, {'.LVCORR': ''}
wave_function, {'.SCF': '', '.RELCCSD': '', '*SCF': {'.PRINT': '1'}, '**RELCCSD': {'.ENERGY': '', '.PRINT': '1', '.FOCKSP': ''}, '*CCFSPC': {'.DOIE2': '', '.NACTH': '1 1 0 0', '.MAXIT': '200'}}
molecule, {'*BASIS': {'.DEFAULT': 'dyall.3zp'}, '*END OF': {}, '*charge': {'.charge': '0'}, '*symmetry': {'.LINEAR': ''}}
integrals, {'.nucmod': '1'}
moltra, {'.active': 'energy -10 20 1'}

 starting DIRAC for molecule dirac.xyz with input dirac.inp
  creating archive file  dirac.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal


In [7]:
ase_molecule.calc.h5_contents

['Group: input',
 'Group: input/aobasis',
 'Group: input/aobasis/1',
 'Dataset: input/aobasis/1/angular (1,)',
 'Dataset: input/aobasis/1/aobasis_id (1,)',
 'Dataset: input/aobasis/1/center (72,)',
 'Dataset: input/aobasis/1/contractions (24,)',
 'Dataset: input/aobasis/1/exponents (24,)',
 'Dataset: input/aobasis/1/n_ao (1,)',
 'Dataset: input/aobasis/1/n_cont (24,)',
 'Dataset: input/aobasis/1/n_prim (24,)',
 'Dataset: input/aobasis/1/n_shells (1,)',
 'Dataset: input/aobasis/1/orbmom (24,)',
 'Group: input/aobasis/2',
 'Dataset: input/aobasis/2/angular (1,)',
 'Dataset: input/aobasis/2/aobasis_id (1,)',
 'Dataset: input/aobasis/2/center (102,)',
 'Dataset: input/aobasis/2/contractions (34,)',
 'Dataset: input/aobasis/2/exponents (34,)',
 'Dataset: input/aobasis/2/n_ao (1,)',
 'Dataset: input/aobasis/2/n_cont (34,)',
 'Dataset: input/aobasis/2/n_prim (34,)',
 'Dataset: input/aobasis/2/n_shells (1,)',
 'Dataset: input/aobasis/2/orbmom (34,)',
 'Dataset: input/dirac_inp (46,)',
 'Group:

In [8]:
n_ao = ase_molecule.calc.extract_h5_value("result/operators/ao_matrices/BETAMAT FFFT")
n_ao

array([0.        , 0.        , 0.        , ..., 0.2301652 , 0.4878467 ,
       1.33938936], shape=(60726,))

In [9]:
ase_molecule.calc.results

{'SCF_energy': -1393.5473329223928,
 'MP2_energy': -148.27627346665378,
 'CCSD_energy': -148.79212401405528,
 'energy': -4048.8399239319815,
 'scf_energy': array([-149.18871037]),
 'scf_eigenvalues': array([-1.57030684e+05, -7.74102831e+04, -5.20712002e+04, -4.29013088e+04,
        -3.95804520e+04, -3.83798113e+04, -3.83628648e+04, -3.82549734e+04,
        -3.82549160e+04, -3.78951841e+04, -3.77872538e+04, -3.77502114e+04,
        -3.77500875e+04, -3.77048580e+04, -3.76417582e+04, -3.76247202e+04,
        -3.76244764e+04, -3.76231048e+04, -3.75913594e+04, -3.75866056e+04,
        -3.75831163e+04, -3.75830649e+04, -3.75741035e+04, -3.75732271e+04,
        -3.75713682e+04, -3.75701398e+04, -3.75697566e+04, -3.75691523e+04,
        -3.75684166e+04, -3.75673873e+04, -3.75670542e+04, -3.75633875e+04,
        -3.75623364e+04, -3.75617728e+04, -3.75617599e+04, -3.75615962e+04,
        -3.75612220e+04, -3.75609953e+04, -3.75606878e+04, -3.75599602e+04,
        -3.75595401e+04, -3.75591090e+04,

Bug in scf energy? 